# Lab Assignment 10
**UCS420 - Cognitive Computing**

NLP using Python - II

In [ ]:
import re
import string
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# downloading the nltk stuff needed for the whole notebook
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

stop_words = set(stopwords.words('english'))

## Q1

In [ ]:
text = ("Cricket is more than just a sport in India, it is almost a festival that brings everyone together. "
        "I still remember watching the 2011 World Cup final with my family, and the whole street went crazy when the winning six was hit. "
        "Players like Virat Kohli and Rohit Sharma train for hours every single day to stay fit and sharp. "
        "T20 matches are fast and exciting, while Test cricket tests the patience and skill of the players. "
        "My friends and I play gully cricket every weekend, even though we have to share one old bat. "
        "Honestly, nothing beats the feeling of hitting a perfect cover drive!")
print(text)

**1. Lowercase + remove punctuation using re**

In [ ]:
clean_text = text.lower()
clean_text = re.sub(r'[^\w\s]', '', clean_text)
print(clean_text)

**2. Tokenizing into words and sentences**

In [ ]:
# sentences are taken from the original text since punctuation is needed to find sentence ends
sentences = sent_tokenize(text)
print("Number of sentences:", len(sentences))
for i, s in enumerate(sentences, 1):
    print(i, s)

words = word_tokenize(clean_text)
print("\nNumber of words:", len(words))
print(words)

**3. split() vs word_tokenize()**

In [ ]:
split_tokens = text.split()
nltk_tokens = word_tokenize(text)

print("split() count       :", len(split_tokens))
print("word_tokenize count :", len(nltk_tokens))

print("\nTokens only in split():")
print([t for t in split_tokens if t not in nltk_tokens])
print("\nTokens only in word_tokenize():")
print([t for t in nltk_tokens if t not in split_tokens])

`split()` only cuts on whitespace, so punctuation stays stuck to the words (like `India,` or `drive!`) and the same word can show up as different tokens. `word_tokenize()` separates the punctuation into its own tokens, so words come out clean, but the punctuation marks are now counted as tokens too (which is why the count is higher). For real NLP work `word_tokenize()` is better, `split()` is fine only for quick and simple cases.

**4. Removing stopwords**

In [ ]:
filtered_words = [w for w in words if w not in stop_words]
print("Before:", len(words), " After:", len(filtered_words))
print(filtered_words)

**5. Word frequency distribution (without stopwords)**

In [ ]:
from nltk.probability import FreqDist

fdist = FreqDist(filtered_words)
print(fdist.most_common(10))

plt.figure(figsize=(10, 4))
fdist.plot(15, title='Top 15 words (stopwords removed)')
plt.show()

## Q2

**1. Only alphabetic words using re.findall()**

In [ ]:
alpha_words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
print(len(alpha_words))
print(alpha_words)

**2. Removing stopwords**

In [ ]:
no_stop = [w for w in alpha_words if w not in stop_words]
print(len(no_stop))
print(no_stop)

**3. Stemming (PorterStemmer)**

In [ ]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()
stemmed = [ps.stem(w) for w in no_stop]
print(stemmed)

**4. Lemmatization (WordNetLemmatizer)**

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

lemma_noun = [lemmatizer.lemmatize(w) for w in no_stop]
lemma_verb = [lemmatizer.lemmatize(w, pos='v') for w in no_stop]
print("As nouns (default):", lemma_noun)
print()
print("As verbs          :", lemma_verb)

**5. Comparison**

In [ ]:
compare = pd.DataFrame({
    'original': no_stop,
    'stemmed': stemmed,
    'lemma (noun)': lemma_noun,
    'lemma (verb)': lemma_verb
})
# showing only the rows where something actually changed
changed = compare[(compare['original'] != compare['stemmed']) | (compare['original'] != compare['lemma (verb)'])]
changed.drop_duplicates().reset_index(drop=True)

Stemming just chops off word endings using fixed rules, so it is very fast but the output is often not a real word (e.g. words ending in `-y` or `-e` get cut to things like `exampl` or `famili`). Lemmatization uses the WordNet dictionary and gives proper dictionary words, but it is slower and it needs the correct POS tag, otherwise it leaves many words unchanged (you can see this in the noun vs verb columns).

**When to use which:**
- Stemming - when speed matters and the exact word doesn't, e.g. search engines, information retrieval, large text classification.
- Lemmatization - when the meaning and readable output matter, e.g. chatbots, sentiment analysis, text generation, question answering.

## Q3

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = [
    "The new phone has an amazing camera and the battery lasts all day long.",
    "Heavy rain in Punjab damaged crops and farmers are waiting for government help.",
    "The movie had great acting but the story was slow and the ending was disappointing."
]
for i, t in enumerate(texts, 1):
    print(f"Text {i}: {t}")

**1. Bag of Words (CountVectorizer)**

In [ ]:
cv = CountVectorizer(stop_words='english')
bow = cv.fit_transform(texts)

bow_df = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out(), index=['Text 1', 'Text 2', 'Text 3'])
bow_df

**2. TF-IDF (TfidfVectorizer)**

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(texts)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray().round(3), columns=tfidf.get_feature_names_out(),
                        index=['Text 1', 'Text 2', 'Text 3'])
tfidf_df

**3. Top 3 keywords from each text**

In [ ]:
feature_names = tfidf.get_feature_names_out()

for i in range(len(texts)):
    row = tfidf_matrix[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:3]
    print(f"Text {i+1} top 3 keywords:")
    for j in top_idx:
        print(f"   {feature_names[j]:<15} {row[j]:.3f}")
    print()

**Interpretation:** Bag of Words only counts how many times a word appears, so every word that appears once looks equally important. TF-IDF fixes this by giving a higher score to words that are specific to one text and a lower score to words that appear in many texts. In my three texts almost every word appears only once, so many words tie on score and the top 3 are picked among tied words. Still, the keywords do point at the topic: `phone` and `battery` for the first text, `rain` and `punjab` for the second, and `story`, `slow` and `movie` for the third. Words like `new` or `waiting` are less meaningful, which shows that TF-IDF on very short texts is not very reliable and works better with longer documents.

## Q4

In [ ]:
ai_text = ("Artificial intelligence is the ability of machines to learn from data and make decisions like humans. "
           "It is used in healthcare, self driving cars and voice assistants. "
           "Machine learning and neural networks are the main parts of modern AI. "
           "AI systems get better as they are trained on more data.")

bc_text = ("Blockchain is a digital ledger that stores data in blocks linked together using cryptography. "
           "It is used in cryptocurrency, supply chains and secure payments. "
           "Every transaction is verified by many computers so no single person controls the data. "
           "Once data is added to the blockchain it is very hard to change.")

print("AI :", ai_text)
print()
print("BC :", bc_text)

**1. Preprocessing and tokenizing**

In [ ]:
def preprocess(t):
    tokens = word_tokenize(t.lower())
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]
    return tokens

ai_tokens = preprocess(ai_text)
bc_tokens = preprocess(bc_text)

print("AI tokens:", ai_tokens)
print()
print("Blockchain tokens:", bc_tokens)

**2a. Jaccard similarity (sets)**

In [ ]:
set_ai = set(ai_tokens)
set_bc = set(bc_tokens)

common = set_ai & set_bc
union = set_ai | set_bc
jaccard = len(common) / len(union)

print("Common words :", common)
print("Intersection :", len(common))
print("Union        :", len(union))
print("Jaccard similarity:", round(jaccard, 4))

**2b. Cosine similarity (TF-IDF)**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

docs = [" ".join(ai_tokens), " ".join(bc_tokens)]
vec = TfidfVectorizer()
mat = vec.fit_transform(docs)

cos_sim = cosine_similarity(mat[0], mat[1])[0][0]
print("Cosine similarity:", round(cos_sim, 4))

**2c. Which one is better here?**

In [ ]:
print(f"Jaccard : {jaccard:.4f}")
print(f"Cosine  : {cos_sim:.4f}")

Both scores come out low, which makes sense because AI and Blockchain are two different technologies and share very few words. Jaccard only looks at whether a word is present or not, and it ignores how often a word is used and how important it is. It also drops quickly when the texts have different lengths. Cosine similarity with TF-IDF takes word frequency and importance into account, so words that are common to everything get less weight, and it is not affected much by the length of the text.

For comparing the meaning/topic of two texts I think cosine similarity with TF-IDF gives the better insight. Jaccard is still useful as a quick and simple check on vocabulary overlap.

## Q5

In [ ]:
reviews = [
    "This phone is absolutely fantastic! The camera quality is superb and the battery lasts really long.",
    "Worst purchase ever. The screen broke in two days and the customer service was rude and unhelpful.",
    "The delivery was on time and the packaging was good. Product works fine.",
    "I love this laptop, it is fast, lightweight and the display looks beautiful. Totally worth the money!",
    "Terrible quality. It stopped working after a week and I am very disappointed.",
    "Great headphones, the sound is clear and bass is amazing. Very comfortable to wear for hours.",
    "The product is okay. Nothing special, nothing bad.",
    "Awful experience, the app keeps crashing and the support team never replied to my emails."
]

**1. Polarity and subjectivity (TextBlob)**

In [ ]:
from textblob import TextBlob

rows = []
for r in reviews:
    blob = TextBlob(r)
    rows.append([r, blob.sentiment.polarity, blob.sentiment.subjectivity])

df = pd.DataFrame(rows, columns=['review', 'polarity', 'subjectivity'])
df[['polarity', 'subjectivity']] = df[['polarity', 'subjectivity']].round(3)
df

**2. Classifying into Positive / Negative / Neutral**

In [ ]:
def classify(p):
    if p > 0.1:
        return 'Positive'
    elif p < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment'] = df['polarity'].apply(classify)
print(df['sentiment'].value_counts())
df[['review', 'polarity', 'subjectivity', 'sentiment']]

**3. Word cloud of positive reviews**

In [ ]:
from wordcloud import WordCloud, STOPWORDS

pos_reviews = " ".join(df[df['sentiment'] == 'Positive']['review'])

wc = WordCloud(width=800, height=400, background_color='white',
               stopwords=STOPWORDS, colormap='viridis').generate(pos_reviews)

plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Positive Reviews')
plt.show()

## Q6

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

train_text = (
    "Machine learning is a branch of artificial intelligence that allows computers to learn from data. "
    "A machine learning model finds patterns in data and uses them to make predictions. "
    "The more data the model sees the better it learns. "
    "Deep learning is a part of machine learning that uses neural networks with many layers. "
    "Neural networks are inspired by the human brain and they can learn complex patterns. "
    "Today machine learning is used in medicine, banking, education and many other fields. "
    "Learning to build good models takes practice and patience."
)
print(len(train_text.split()), "words in training text")

**1. Tokenizer**

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([train_text])
total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)
print(list(tokenizer.word_index.items())[:15])

**2. Input sequences and model**

In [ ]:
# making n-gram sequences line by line
input_sequences = []
for line in train_text.split('.'):
    line = line.strip()
    if not line:
        continue
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_len = max(len(s) for s in input_sequences)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_len, padding='pre'))
print("Total sequences:", len(input_sequences), " | max length:", max_len)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)
print(X.shape, y.shape)

In [ ]:
model = Sequential([
    Embedding(total_words, 32),
    LSTM(100),
    Dense(total_words, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.build(input_shape=(None, max_len - 1))
model.summary()

**3. Training and generating text**

In [ ]:
history = model.fit(X, y, epochs=300, verbose=0)
print("Final accuracy:", round(history.history['accuracy'][-1], 3))

plt.plot(history.history['loss'])
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss')
plt.show()

In [ ]:
def generate_text(seed, n_words=8):
    result = seed
    for _ in range(n_words):
        seq = tokenizer.texts_to_sequences([result])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding='pre')
        pred = model.predict(seq, verbose=0)
        next_idx = int(np.argmax(pred, axis=-1)[0])
        next_word = tokenizer.index_word.get(next_idx, '')
        result += ' ' + next_word
    return result

for seed in ['machine', 'neural', 'deep learning']:
    print(generate_text(seed))